# Telecom Customer Churn — Statistical Testing

**Objective:** Check whether customer characteristics are associated with churn and compare numerical measures between churned and retained customers.

This notebook includes Chi-square tests for categorical variables, Mann–Whitney U tests for numerical variables, and Holm correction for multiple testing.

**Important:** Statistical association does not establish causation.

## 1. Import libraries and load data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import chi2_contingency, mannwhitneyu
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", None)

data_path = Path("../data/processed/telco_churn_cleaned.csv")
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nChurn counts:")
print(df["Churn"].value_counts())
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

Dataset shape: (7043, 21)

Churn counts:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Missing values:
TotalCharges    11
dtype: int64


## 2. Prepare analysis columns

Create the tenure groups used in the earlier analysis. The 11 missing `TotalCharges` values are retained and excluded only from calculations that require that column.

In [2]:
def tenure_group(tenure):
    if tenure <= 12:
        return "New (0-12 months)"
    elif tenure <= 24:
        return "Early (13-24 months)"
    elif tenure <= 48:
        return "Mid-term (25-48 months)"
    return "Long-term (49-72 months)"

df["TenureGroup"] = df["tenure"].apply(tenure_group)

print(df[["tenure", "TenureGroup", "Churn"]].head())

   tenure              TenureGroup Churn
0       1        New (0-12 months)    No
1      34  Mid-term (25-48 months)    No
2       2        New (0-12 months)   Yes
3      45  Mid-term (25-48 months)    No
4       2        New (0-12 months)   Yes


## 3. Chi-square tests — categorical variables

For each variable, the Chi-square test checks whether its distribution is independent of churn.

- **Null hypothesis:** The variable and churn are independent.
- **Alternative hypothesis:** There is an association between the variable and churn.
- Significance level: 0.05.

The **strength-of-association** value is Cramér's V. It ranges from 0 to 1; larger values indicate a stronger association. Interpret it in context rather than using rigid universal cut-offs.

In [3]:
categorical_variables = [
    "Contract",
    "TenureGroup",
    "PaymentMethod",
    "InternetService",
    "OnlineSecurity",
    "TechSupport",
    "PaperlessBilling",
    "SeniorCitizen"
]

chi_square_results = []
contingency_tables = {}
expected_frequency_checks = {}

for variable in categorical_variables:
    table = pd.crosstab(df[variable], df["Churn"])
    contingency_tables[variable] = table

    chi2, p_value, dof, expected = chi2_contingency(table)
    n = table.to_numpy().sum()
    min_dimension = min(table.shape[0] - 1, table.shape[1] - 1)
    strength = np.sqrt(chi2 / (n * min_dimension))

    expected_frequency_checks[variable] = {
        "Minimum expected count": expected.min(),
        "All expected counts >= 5": bool((expected >= 5).all())
    }

    chi_square_results.append({
        "Variable": variable,
        "Chi-square statistic": chi2,
        "Degrees of freedom": dof,
        "P-value": p_value,
        "Strength of association": strength
    })

chi_square_results_df = pd.DataFrame(chi_square_results)
chi_square_results_df

,Variable,Chi-square statistic,Degrees of freedom,P-value,Strength of association
0,Contract,1184.596572,2,5.863038e-258,0.410116
1,TenureGroup,856.100239,3,2.943770e-185,0.348645
2,PaymentMethod,648.142327,3,3.682355e-140,0.303359
3,InternetService,732.309590,2,9.571788e-160,0.322455
4,OnlineSecurity,849.998968,2,2.661150e-185,0.347400
5,TechSupport,828.197068,2,1.443084e-180,0.342916
6,PaperlessBilling,258.277649,1,4.073355e-58,0.191498
7,SeniorCitizen,159.426300,1,1.510067e-36,0.150453


### 3.1 Expected-frequency check

In [4]:
expected_checks_df = pd.DataFrame(expected_frequency_checks).T
expected_checks_df

,Minimum expected count,All expected counts >= 5
Contract,390.88982,True
TenureGroup,271.738748,True
PaymentMethod,403.892943,True
InternetService,404.954423,True
OnlineSecurity,404.954423,True
TechSupport,404.954423,True
PaperlessBilling,762.142269,True
SeniorCitizen,303.052392,True


A p-value below 0.05 indicates evidence of an association in this sample. Check expected counts before relying on a Chi-square result. If any expected count is below 5, review the table and consider an appropriate alternative.

## 4. Mann–Whitney U tests — numerical variables

Compare churned and retained customers for tenure, MonthlyCharges, and TotalCharges.

The Mann–Whitney U test compares the distributions of two independent groups. It does not directly test equality of means. Missing values are excluded separately for each variable.

- **Null hypothesis:** The two groups have the same distribution.
- **Alternative hypothesis:** Their distributions differ.

In [5]:
numeric_variables = ["tenure", "MonthlyCharges", "TotalCharges"]
mann_whitney_results = []

for variable in numeric_variables:
    retained = df.loc[df["Churn"] == "No", variable].dropna()
    churned = df.loc[df["Churn"] == "Yes", variable].dropna()

    statistic, p_value = mannwhitneyu(
        retained, churned, alternative="two-sided"
    )

    mann_whitney_results.append({
        "Variable": variable,
        "Retained count": len(retained),
        "Churned count": len(churned),
        "Retained median": retained.median(),
        "Churned median": churned.median(),
        "Mann-Whitney U statistic": statistic,
        "P-value": p_value
    })

mann_whitney_results_df = pd.DataFrame(mann_whitney_results)
mann_whitney_results_df

,Variable,Retained count,Churned count,Retained median,Churned median,Mann-Whitney U statistic,P-value
0,tenure,5174,1869,38.000,10.00,7154668.0,2.419636e-208
1,MonthlyCharges,5174,1869,64.425,79.65,3667080.5,3.311628e-54
2,TotalCharges,5163,1869,1683.600,703.55,6288982.0,1.995985e-84


## 5. Multiple-testing correction

Testing several variables increases the chance of a small p-value occurring by chance. Apply the Holm correction across all categorical and numerical tests. Use the adjusted p-values for the final significance decision at 0.05.

In [6]:
all_test_results = pd.concat([
    chi_square_results_df[["Variable", "P-value"]].assign(Test="Chi-square"),
    mann_whitney_results_df[["Variable", "P-value"]].assign(Test="Mann-Whitney U")
], ignore_index=True)

reject, adjusted_p_values, _, _ = multipletests(
    all_test_results["P-value"],
    alpha=0.05,
    method="holm"
)

all_test_results["Holm-adjusted p-value"] = adjusted_p_values
all_test_results["Significant after Holm correction"] = reject

all_test_results.sort_values("Holm-adjusted p-value")

,Variable,P-value,Test,Holm-adjusted p-value,Significant after Holm correction
0,Contract,5.863038e-258,Chi-square,6.449342e-257,True
8,tenure,2.419636e-208,Mann-Whitney U,2.419636e-207,True
4,OnlineSecurity,2.661150e-185,Chi-square,2.395035e-184,True
1,TenureGroup,2.943770e-185,Chi-square,2.395035e-184,True
5,TechSupport,1.443084e-180,Chi-square,1.010159e-179,True
3,InternetService,9.571788e-160,Chi-square,5.743073e-159,True
2,PaymentMethod,3.682355e-140,Chi-square,1.841177e-139,True
10,TotalCharges,1.995985e-84,Mann-Whitney U,7.983940e-84,True
6,PaperlessBilling,4.073355e-58,Chi-square,1.222006e-57,True
9,MonthlyCharges,3.311628e-54,Mann-Whitney U,6.623255e-54,True


## 6. Findings and business interpretation

Use the result tables to identify which variables remain statistically significant after Holm correction and which categorical variables have the strongest association with churn.

For numerical variables, compare group medians and distributions alongside adjusted p-values.

**Business caution:** These are observational results. They identify patterns worth investigating; they do not show that changing a contract, payment method, or service will necessarily reduce churn.

In [7]:
categorical_summary = chi_square_results_df.merge(
    all_test_results[["Variable", "Holm-adjusted p-value",
                      "Significant after Holm correction"]],
    on="Variable",
    how="left"
)

categorical_summary.sort_values(
    "Strength of association", ascending=False
)

,Variable,Chi-square statistic,Degrees of freedom,P-value,Strength of association,Holm-adjusted p-value,Significant after Holm correction
0,Contract,1184.596572,2,5.863038e-258,0.410116,6.449342e-257,True
1,TenureGroup,856.100239,3,2.943770e-185,0.348645,2.395035e-184,True
4,OnlineSecurity,849.998968,2,2.661150e-185,0.347400,2.395035e-184,True
5,TechSupport,828.197068,2,1.443084e-180,0.342916,1.010159e-179,True
3,InternetService,732.309590,2,9.571788e-160,0.322455,5.743073e-159,True
2,PaymentMethod,648.142327,3,3.682355e-140,0.303359,1.841177e-139,True
6,PaperlessBilling,258.277649,1,4.073355e-58,0.191498,1.222006e-57,True
7,SeniorCitizen,159.426300,1,1.510067e-36,0.150453,1.510067e-36,True
